# TCC - Analise Preditiva de Risco Academico Escolar
## CRISP-DM: 1. Business Understanding

**Autor:** Warley Rodrigues de Andrade  
**Curso:** Sistemas de Informacao  
**Instituicao:** Instituto Federal de Goias  
**Orientador:** Otavio Xavier Calaca  
**Ano:** 2026

Este notebook apresenta a primeira fase do CRISP-DM aplicada ao projeto: o entendimento do negocio. Nesta fase, o foco e definir o problema institucional, os usuarios, o valor pedagogico esperado, os alvos preditivos e os criterios de sucesso da solucao, sem expor dados sensiveis nem detalhes privados do banco.

## Escopo desta serie CRISP-DM

Esta serie organiza a demonstracao do TCC nas seis fases classicas do CRISP-DM:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

Neste item 1, o objetivo e esclarecer o problema e definir o que sera considerado sucesso. Nas proximas fases, a demonstracao deve continuar usando apenas amostras anonimizadas, agregadas ou sinteticas, sem expor SQL real, credenciais ou a estrutura completa do banco institucional.

## Contexto do problema

Em uma instituicao escolar, professores, coordenadores e secretaria acompanham informacoes de alunos em diferentes frentes: notas, faltas, situacao de matricula, turma, serie, curso, pagamentos e vinculos com professores. O problema e que esses sinais normalmente sao analisados de forma separada e reativa.

Na pratica, muitas intervencoes acontecem apenas depois que o aluno ja apresenta queda relevante de desempenho, excesso de faltas ou risco de reprovacao. O projeto busca antecipar esses sinais, transformando historico escolar em relatorios que apoiem a tomada de decisao pedagogica.

## Problema de negocio

A escola precisa identificar, com antecedencia, quais alunos podem apresentar piora academica em uma disciplina ou entrar em uma faixa de risco. Essa identificacao nao deve depender apenas da ultima nota observada, porque o desempenho do aluno pode ser influenciado por historico anterior, tendencia recente, faltas acumuladas, diferenca em relacao a turma, quantidade de etapas restantes, contexto da serie e outros fatores.

O desafio do projeto e transformar dados historicos em uma visao operacional simples: quem precisa de atencao, em qual disciplina, em qual turma, com qual nivel de risco e com quais sinais justificando esse alerta. Em outras palavras, a escola nao quer apenas constatar a reprovacao; ela quer agir antes.

### Exemplo fixo do problema de negocio

| papel | situacao | decisao |
| --- | --- | --- |
| Professor | queda recente em Matematica | intervencao pedagogica antes da proxima etapa |
| Coordenacao | turma com varios alunos em risco alto | priorizar acompanhamento por serie e disciplina |
| Secretaria | faltas e pendencias se acumulando | acionar familia e apoiar regularizacao |

## Objetivo geral

Construir uma pipeline preditiva capaz de estimar a proxima nota do aluno e indicar risco academico, gerando relatorios, rankings e visualizacoes operacionais para professores, coordenacao e secretaria com foco em acompanhamento pedagogico preventivo.

## Objetivos especificos

- Extrair dados escolares tratados a partir de CSVs canonicos, sem expor as consultas SQL reais.
- Unificar notas, faltas, pagamentos e vinculos de professor em uma base temporal de modelagem.
- Criar atributos historicos por aluno, disciplina, etapa, serie e ano letivo.
- Prever a proxima nota do aluno por regressao.
- Classificar se a proxima nota tende a ficar abaixo da linha de risco pedagogico.
- Comparar modelos de aprendizado de maquina com baselines simples, como ultima nota e medias historicas, usando metricas coerentes com cada tarefa.
- Gerar relatorios finais por perfil de uso: professor, coordenacao e secretaria.
- Disponibilizar visualizacao operacional em dashboard Streamlit.

## Usuarios e interessados

| Perfil | Interesse principal | Exemplo de uso |
|---|---|---|
| Professor | Ver alunos com risco em suas disciplinas e turmas | Priorizar revisao, reforco ou acompanhamento individual |
| Coordenacao | Identificar concentracao de risco por serie, turma, disciplina e professor | Planejar intervencoes pedagogicas e reunioes de acompanhamento |
| Secretaria | Apoiar visao cadastral, historica e operacional | Acompanhar situacoes que exigem contato, registro ou verificacao |
| Gestao escolar | Entender padroes gerais de desempenho e risco | Avaliar a necessidade de acoes institucionais |

## Perguntas de negocio

- Qual tende a ser a proxima nota de um aluno em determinada disciplina?
- O aluno esta caminhando para uma nota abaixo da linha de risco academico?
- Quais alunos devem receber atencao primeiro?
- Quais disciplinas, series ou turmas concentram maior erro ou maior risco?
- Quais professores possuem maior quantidade de alunos em risco, considerando suas turmas e disciplinas?
- Qual e o minimo de historico necessario para gerar uma avaliacao mais confiavel?
- O modelo esta realmente superando regras simples, como repetir a ultima nota como previsao?

## Tipo de problema de ciencia de dados

O projeto possui dois objetivos tecnicos complementares:

| Objetivo | Tipo de tarefa | Alvo previsto | Exemplo |
|---|---|---|---|
| Previsao de nota | Regressao | `target_nota_proxima` | prever que a proxima media em Matematica sera 5.8 |
| Alerta de risco | Classificacao binaria | `target_risco_proxima` | indicar risco quando a proxima nota esperada ou historicamente observada fica abaixo de 6.0 |

A regressao responde a pergunta 'qual nota provavelmente vem a seguir?'. A classificacao responde a pergunta operacional 'o aluno precisa de atencao por risco academico?'.

## Exemplo conceitual do alvo de previsao

Imagine um aluno com as seguintes notas em Lingua Portuguesa:

| Ano | Etapa | Nota observada | Proxima nota usada como alvo |
|---|---:|---:|---:|
| 2025 | 1 | 7.0 | 6.2 |
| 2025 | 2 | 6.2 | 5.5 |
| 2025 | 3 | 5.5 | 6.0 |

Na linha da etapa 2, por exemplo, o modelo enxerga o historico disponivel ate aquele momento e tenta prever a etapa seguinte. Se a proxima nota real for 5.5, o alvo de regressao daquela linha e 5.5. O alvo de classificacao sera 1, porque 5.5 esta abaixo de 6.0.

### Exemplo fixo do alvo de previsao

| NomeSerie | NomeDisciplina | nota_atual | faltas_acumuladas | pagamentos_pendentes_ano | target_nota_proxima | target_risco_proxima |
| --- | --- | --- | --- | --- | --- | --- |
| 9º Ano | Matematica 2 | 4.0 | 8 | 2 | 3.5 | 1 |
| 1ª Serie | Quimica 1 | 7.4 | 1 | 0 | 7.1 | 0 |

## Fontes de dados esperadas

O repositorio publico nao depende diretamente do desenho completo do banco institucional. A interface publica da pipeline sao CSVs canonicos em `artifacts/database/csv/`, conforme o contrato descrito em `docs/ENTRADA_DE_DADOS_E_CONTRATOS.md`:

- `aluno.csv`: vinculos escolares do aluno.
- `media_nota_aluno.csv`: notas e medias por disciplina e etapa.
- `faltas_aluno.csv`: eventos de falta por disciplina e etapa.
- `pagamento_aluno.csv`: informacoes financeiras agregaveis por aluno e ano.
- `responsaveis_aluno.csv`: vinculos de responsaveis, quando usados em relatorios.
- `professor_disciplina.csv`: vinculo entre professor, turma e disciplina quando existir.

As consultas SQL reais e a preparacao fisica do banco ficam fora do Git, em camada privada local.

## Criterios de sucesso tecnico

| Parte | Metrica principal | Interpretacao |
|---|---|---|
| Previsao de nota | MAE | erro medio absoluto entre nota prevista e nota real |
| Previsao de nota | RMSE | penaliza erros grandes com mais intensidade |
| Previsao de nota | acerto ate 0.5 ponto | porcentagem de previsoes com erro maximo de meio ponto |
| Alerta de risco | Precision | quando o modelo alerta risco, quantas vezes estava correto |
| Alerta de risco | Recall | entre os alunos que realmente entraram em risco, quantos foram encontrados |
| Alerta de risco | F1 | equilibrio entre precision e recall |

Na implementacao atual, a selecao principal usa MAE para regressao e F1 para classificacao, enquanto RMSE, acerto ate 0.5 ponto, precision e recall ajudam a interpretar o comportamento do modelo. No contexto escolar, um bom modelo nao deve ser avaliado apenas por uma porcentagem geral: ele precisa errar pouco nas notas e tambem identificar bem alunos em risco, porque falso negativo pode significar deixar de apoiar um aluno que precisava de intervencao.

## Criterios de sucesso pedagogico

- O professor consegue entender quais alunos priorizar.
- A coordenacao consegue enxergar concentracoes de risco por turma, serie, disciplina e professor.
- A secretaria consegue apoiar a leitura operacional sem depender de planilhas isoladas.
- O projeto nao expõe dados pessoais nem consultas internas do banco.
- Os relatorios ajudam na acao preventiva, nao apenas na constatacao tardia de reprovacao.

## Restricoes e cuidados

- Dados escolares sao sensiveis e devem permanecer anonimizados fora do ambiente autorizado.
- Alunos podem entrar e sair da escola, portanto nem todos terao historico suficiente.
- Turmas e disciplinas podem existir sem professor vinculado; isso deve ser tratado como ausencia legitima de dado, nao como erro.
- A avaliacao precisa respeitar a ordem temporal, treinando com anos anteriores e testando em anos posteriores.
- O modelo deve ser usado como apoio a decisao, nao como decisao automatica sobre o aluno.

## Resultado esperado ao final do projeto

Ao final da pipeline, espera-se gerar:

- datasets de modelagem em `artifacts/pipeline/`;
- relatorios tecnicos de previsao e risco;
- analises de erro por disciplina, serie e faixa de nota;
- relatorios finais para professor, coordenacao e secretaria;
- ranking executivo de alunos prioritarios e ranking de professores por volume de risco;
- dashboard Streamlit para visualizacao dos principais indicadores.

## Decisao desta fase

O projeto deve seguir para a fase de Data Understanding usando apenas dados ja extraidos, tratados e anonimizados. A proxima etapa deve mostrar o formato dos CSVs, suas colunas, quantidade de registros, periodos disponiveis, exemplos agregados, chaves de integracao e primeiros problemas encontrados, sem revelar dados pessoais nem SQL institucional.

## Saida didatica da fase

Nesta fase ainda nao existe transformacao fisica dos dados nem treinamento de modelo. O que muda aqui e a definicao do problema que guiará todas as etapas seguintes.

Quando este item termina, o projeto ja deixou claro:
- qual pergunta institucional quer responder;
- quais dados precisara integrar depois;
- qual sera o alvo da regressao e o alvo da classificacao;
- quais metricas dirao se a solucao ficou util;
- quais perfis da escola receberao as entregas finais.

Ou seja, a principal saida do Business Understanding nao e um arquivo de dados novo, mas um contrato de decisao: o que o projeto deve prever, para quem, com que criterio de sucesso e com quais restricoes de seguranca.

### Exemplo fixo da saida desta fase

| fase | saida | uso |
| --- | --- | --- |
| Business Understanding | problema definido | alinhar o que a escola quer prevenir |
| Data Understanding | mapa das bases | saber o que existe e o que falta |
| Deployment | relatorios e dashboard | transformar previsoes em acao escolar |